In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import plotly.express as px

In [3]:
df = pd.read_csv("../../data/cardekho_imputated.csv", index_col=[0])

In [4]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [ ]:
"""

Data Cleaning:

1) handling missing values
2) handling duplicates
3) checking datatype
4) understanding dataset

"""

In [ ]:
# checling null values

df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [8]:
# removing unnecessay columns

df.drop(columns=["car_name", "brand"], inplace=True)

In [9]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [10]:
df["model"].unique()

<StringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [11]:
# classifying different types of features

num_features = [feature for feature in df.columns if df[feature].dtype in ["float64", "int64"]]
print("Count of numerical features:", len(num_features))

cat_features = [feature for feature in df.columns if df[feature].dtype == "str"]
print("Count of categorical features:", len(cat_features))

discrete_features = [feature for feature in num_features if len(df[feature].unique()) <= 25]
print("Count of discrete features:", len(discrete_features))

continuous_features = [feature for feature in num_features if feature not in discrete_features]
print("Count of continuous features:", len(continuous_features))

Count of numerical features: 7
Count of categorical features: 4
Count of discrete features: 2
Count of continuous features: 5


In [12]:
# defining independent and dependent features

x = df.drop(["selling_price"], axis=1)
y = df["selling_price"]

In [13]:
x.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [14]:
# feature encoding and scaling

In [15]:
df["model"].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

x["model"] = le.fit_transform(x["model"])

In [17]:
x.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [20]:
len(df["seller_type"].unique()), len(df["fuel_type"].unique()), len(df["transmission_type"].unique())

(3, 5, 2)

In [23]:
# creating column transformer with 3 types of transformers

num_features = [feature for feature in df.columns if df[feature].dtype in ["float64", "int64"]]
onehot_columns = ["seller_type", "fuel_type", "transmission_type"]

num_features.remove("selling_price")

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

scaler = StandardScaler()
oh_transformer = OneHotEncoder(drop="first")

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, onehot_columns),
        ("StandardScaler", scaler, num_features)
    ],
    remainder="passthrough",
)

In [24]:
x = preprocessor.fit_transform(x)

In [25]:
pd.DataFrame(x)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022,7.0
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022,54.0
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022,118.0
3,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022,7.0
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022,38.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022,117.0
15407,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444,42.0
15408,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022,77.0
15409,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444,114.0


In [26]:
# train test split

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [27]:
x_train.shape, x_test.shape

((12328, 14), (3083, 14))

In [28]:
# model training

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [29]:
def evaluate_model(true, pred):
    
    mae = mean_absolute_error(true, pred)
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)

    score = r2_score(true, pred)

    return mae, rmse, score

In [30]:
models = {
    "Linear Regression" : LinearRegression(),
    "Lasso" : Lasso(),
    "Ridge" : Ridge(),
    "Decision Tree" : DecisionTreeRegressor(),
    "Random Forest" : RandomForestRegressor(),
}

In [31]:

for i in range(len(list(models))):

    model = list(models.values())[i]

    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred  = model.predict(x_test)

    # performance on training data

    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    # performance on testing  data

    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)


    print("\n", list(models.keys())[i])

    print("\nModel Performance on Training Set:")
    print("\tRMSE: \t\t\t", {model_train_rmse})
    print("\tMAE: \t\t\t", {model_train_mae})
    print("\tR2 Score: \t\t\t", {model_train_r2})

    print("\nModel Performance on Testing Set:")
    print("\tRMSE: \t\t\t", {model_test_rmse})
    print("\tMAE: \t\t\t", {model_test_mae})
    print("\tR2 Score: \t\t\t", {model_test_r2})



 Linear Regression

Model Performance on Training Set:
	RMSE: 			 {np.float64(553855.6665411664)}
	MAE: 			 {268101.607082986}
	R2 Score: 			 {0.6217719576765959}

Model Performance on Testing Set:
	RMSE: 			 {np.float64(502543.59302309627)}
	MAE: 			 {279618.5794158351}
	R2 Score: 			 {0.6645109298852034}

 Lasso

Model Performance on Training Set:
	RMSE: 			 {np.float64(553855.6709529602)}
	MAE: 			 {268099.31921294413}
	R2 Score: 			 {0.6217719516509677}

Model Performance on Testing Set:
	RMSE: 			 {np.float64(502542.7116635146)}
	MAE: 			 {279614.84056292445}
	R2 Score: 			 {0.6645121066438022}

 Ridge

Model Performance on Training Set:
	RMSE: 			 {np.float64(553856.3159177309)}
	MAE: 			 {268059.9525076336}
	R2 Score: 			 {0.6217710707575461}

Model Performance on Testing Set:
	RMSE: 			 {np.float64(502533.8886790387)}
	MAE: 			 {279557.36472942546}
	R2 Score: 			 {0.6645238866514407}

 Decision Tree

Model Performance on Training Set:
	RMSE: 			 {np.float64(20797.23516567643)}

In [32]:
# hyperparameter tuning

In [33]:
knn_params = {
    "n_neighbors" : [2, 3, 10, 20, 40, 50]
}

rf_params = {
    "max_depth" : [5, 8, 10, 15, None],
    "max_features" : [5, 7, 8, "auto"],
    "min_samples_split" : [2, 8, 15, 20],
    "n_estimators" : [100, 200, 500, 1000],
}

In [35]:
# models

randomCV_models = [
    ("KNN", KNeighborsRegressor(), knn_params),
    ("RF", RandomForestRegressor(), rf_params),
]

In [36]:
from sklearn.model_selection import RandomizedSearchCV

model_params = {}

for name, model, params in randomCV_models:

    random = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=100,
        cv=3,
        verbose=2,
        n_jobs=-1,
    )

    random.fit(x_train, y_train)

    model_params[name] = random.best_params_

for model_name in model_params:

    print(f"Best params for {model_name}: ")
    print(model_params[model_name])


c:\Users\Bhargav Ram\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 6 is smaller than n_iter=100. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits


c:\Users\Bhargav Ram\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
72 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
32 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Bhargav Ram\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Bhargav Ram\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py", line 1329, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~

Best params for KNN: 
{'n_neighbors': 2}
Best params for RF: 
{'n_estimators': 500, 'min_samples_split': 2, 'max_features': 8, 'max_depth': None}


In [40]:
models = {
    "KNN Regressor" : KNeighborsRegressor(
        n_neighbors=2,
        n_jobs=-1,
    ),
    "RF Regressor" : RandomForestRegressor(
        n_estimators=500,
        min_samples_split=2,
        max_depth=None,
        max_features=8,
    )
}

In [41]:

for i in range(len(list(models))):

    model = list(models.values())[i]

    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred  = model.predict(x_test)

    # performance on training data

    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    # performance on testing  data

    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)


    print("\n", list(models.keys())[i])

    print("\nModel Performance on Training Set:")
    print("\tRMSE: \t\t\t", {model_train_rmse})
    print("\tMAE: \t\t\t", {model_train_mae})
    print("\tR2 Score: \t\t\t", {model_train_r2})

    print("\nModel Performance on Testing Set:")
    print("\tRMSE: \t\t\t", {model_test_rmse})
    print("\tMAE: \t\t\t", {model_test_mae})
    print("\tR2 Score: \t\t\t", {model_test_r2})



 KNN Regressor

Model Performance on Training Set:
	RMSE: 			 {np.float64(179585.6137870367)}
	MAE: 			 {61203.15541855938}
	R2 Score: 			 {0.9602347602883046}

Model Performance on Testing Set:
	RMSE: 			 {np.float64(255478.8061960967)}
	MAE: 			 {110568.15601686669}
	R2 Score: 			 {0.9132956970525463}

 RF Regressor

Model Performance on Training Set:
	RMSE: 			 {np.float64(128106.9065342819)}
	MAE: 			 {38981.348564975066}
	R2 Score: 			 {0.9797648882817938}

Model Performance on Testing Set:
	RMSE: 			 {np.float64(215399.53842215496)}
	MAE: 			 {98210.74192258688}
	R2 Score: 			 {0.9383659890036722}
